### Setup AWS Credentials

In [1]:
import os
# Setup your AWS Access Key and Secret Key as environment variables.
#os.environ["AWS_ACCESS_KEY_ID"]
#os.environ["AWS_SECRET_ACCESS_KEY"] 

import sys
sys.path.insert(0, '../../../src')


In [2]:
# Setup Nova Model
NOVA_MODEL_ID = "us.amazon.nova-pro-v1:0"

### Dataset Adapter

Initialize the Dataset Adapter that takes the input_columns and output_columns. We use the CSVDatasetAdapter to read a `.csv` file and adapt it to the standardized format. We also use the adapter to create train and test sets for our use case.

In [3]:
from amzn_nova_prompt_optimizer.core.input_adapters.dataset_adapter import CSVDatasetAdapter

input_columns = {"input"}
output_columns = {"answer"}

dataset_adapter = CSVDatasetAdapter(input_columns, output_columns)

# Adapt
dataset_adapter.adapt("../data/FacilitySupportAnalyzer.csv")

train_set, test_set = dataset_adapter.split(0.5)

### Prompt Adapter

Initialize the Prompt Adapter for the Original Prompt. For this example, we use the FacilitySupportAnalyzer System and User Prompt in the `.txt` format. 

In [4]:
from amzn_nova_prompt_optimizer.core.input_adapters.prompt_adapter import TextPromptAdapter

prompt_variables = input_columns

prompt_adapter = TextPromptAdapter()

prompt_adapter.set_system_prompt(file_path="original_prompt/system_prompt.txt", variables=prompt_variables)
prompt_adapter.set_user_prompt(file_path="original_prompt/user_prompt.txt", variables=prompt_variables)

# Adapt
prompt_adapter.adapt()

### Metric Adapter

Initialize the Metric Adapter for evaluating this prompt for certain optimizers. For this example, we build a Custom Metric for the FacilitySupportAnalyzer Dataset. The metric adapter requires the use of the `apply` [For single row evaluation] or `batch_apply` [For evaluating the whole dataset together] function

In [5]:
from amzn_nova_prompt_optimizer.core.input_adapters.metric_adapter import MetricAdapter
from typing import List, Any, Dict
import re
import json

class FacilitySupportAnalyzerMetric(MetricAdapter):
    def parse_json(self, input_string: str):
        """
        Attempts to parse the given string as JSON. If direct parsing fails,
        it tries to extract a JSON snippet from code blocks formatted as:
            ```json
            ... JSON content ...
            ```
        or any code block delimited by triple backticks and then parses that content.
        """
        try:
            return json.loads(input_string)
        except json.JSONDecodeError as err:
            error = err

        patterns = [
            re.compile(r"```json\s*(.*?)\s*```", re.DOTALL | re.IGNORECASE),
            re.compile(r"```(.*?)```", re.DOTALL)
        ]

        for pattern in patterns:
            match = pattern.search(input_string)
            if match:
                json_candidate = match.group(1).strip()
                try:
                    return json.loads(json_candidate)
                except json.JSONDecodeError:
                    continue

        raise error

    def _calculate_metrics(self, y_pred: Any, y_true: Any) -> Dict:
        strict_json = False
        result = {
            "is_valid_json": False,
            "correct_categories": 0.0,
            "correct_sentiment": False,
            "correct_urgency": False,
        }

        try:
            y_true = y_true if isinstance(y_true, dict) else (json.loads(y_true) if strict_json else self.parse_json(y_true))
            y_pred = y_pred if isinstance(y_pred, dict) else (json.loads(y_pred) if strict_json else self.parse_json(y_pred))
        except json.JSONDecodeError:
            result["total"] = 0
            return result  # Return result with is_valid_json = False
        else:
            if isinstance(y_pred, str):
                result["total"] = 0
                return result  # Return result with is_valid_json = False
            result["is_valid_json"] = True

            categories_true = y_true.get("categories", {})
            categories_pred = y_pred.get("categories", {})

            if isinstance(categories_true, dict) and isinstance(categories_pred, dict):
                correct = sum(
                    categories_true.get(k, False) == categories_pred.get(k, False)
                    for k in categories_true
                )
                result["correct_categories"] = correct / len(categories_true) if categories_true else 0.0
            else:
                result["correct_categories"] = 0.0  # or raise an error if you prefer

            result["correct_sentiment"] = y_pred.get("sentiment", "") == y_true.get("sentiment", "")
            result["correct_urgency"] = y_pred.get("urgency", "") == y_true.get("urgency", "")

        # Compute overall metric score
        result["total"] = sum(
            float(result[k]) for k in ["correct_categories", "correct_sentiment", "correct_urgency"]
        ) / 3.0

        return result

    def apply(self, y_pred: Any, y_true: Any):
        return self._calculate_metrics(y_pred, y_true)

    def batch_apply(self, y_preds: List[Any], y_trues: List[Any]):
        evals = [self.apply(y_pred, y_true) for y_pred, y_true in zip(y_preds, y_trues)]
        float_keys = [k for k, v in evals[0].items() if isinstance(v, (int, float, bool))]
        return {k: sum(e[k] for e in evals) / len(evals) for k in float_keys}

metric_adapter = FacilitySupportAnalyzerMetric()

### Inference Adapter
Initialize the InferenceAdapter to choose the backend Inference. Currently, we only support BedrockInferenceAdapter.

In [6]:
from amzn_nova_prompt_optimizer.core.inference import BedrockInferenceAdapter

inference_adapter = BedrockInferenceAdapter(region_name="us-east-1")

### Evaluator

The Evaluator can use the metric_adapter, prompt_adapter, and dataset_adapter to evaluate the prompt given the `model_id` to produce an evaluation score. The Evaluator internally uses the `InferenceRunner` to first generate inference results and then evaluate the output.

#### Base Model Evaluation

In [7]:
from amzn_nova_prompt_optimizer.core.evaluation import Evaluator

evaluator = Evaluator(prompt_adapter, test_set, metric_adapter, inference_adapter)

In [8]:
original_prompt_score = evaluator.aggregate_score(model_id=NOVA_MODEL_ID)

print(f"Original Prompt Evaluation Score = {original_prompt_score}")

2026/01/29 18:19:53 INFO amzn_nova_prompt_optimizer.core.evaluation: Cache miss - Running new inference on Dataset
Running inference: 100%|██████████| 100/100 [00:39<00:00,  2.55it/s]
2026/01/29 18:20:32 INFO amzn_nova_prompt_optimizer.core.evaluation: Running Batch Evaluation on Dataset, using `batch_apply` metric
2026/01/29 18:20:32 INFO amzn_nova_prompt_optimizer.core.evaluation: Using cached inference results
2026/01/29 18:20:32 INFO amzn_nova_prompt_optimizer.core.evaluation: Running Evaluation on Dataset, using `apply` metric


Original Prompt Evaluation Score = {'is_valid_json': 0.24, 'correct_categories': 0.22099999999999997, 'correct_sentiment': 0.12, 'correct_urgency': 0.21, 'total': 0.18366666666666667}


### Optimization Adapter

We can now define the Optimization Functions. The Optimization function takes as input the Prompt Adapter and Optionally a Dataset Adapter, Inference Adapter, and Metric Adapter. The optimization function optimizes the prompt and returns a Prompt Adapter.

In [9]:
class FacilitySupportAnalyzerNovaMetric(FacilitySupportAnalyzerMetric):
    def apply(self, y_pred: Any, y_true: Any):
        # Requires to return a value and not a JSON payload
        return self._calculate_metrics(y_pred, y_true)["total"]
        
    def batch_apply(self, y_preds: List[Any], y_trues: List[Any]):
        pass
nova_metric_adapter = FacilitySupportAnalyzerNovaMetric()

#### NovaPromptOptimizer

NovaPromptOptimizer = Nova Meta Prompter + MIPROv2 with Nova Model Tips

In [10]:
from amzn_nova_prompt_optimizer.core.optimizers import NovaPromptOptimizer

nova_prompt_optimizer = NovaPromptOptimizer(prompt_adapter=prompt_adapter, inference_adapter=inference_adapter, dataset_adapter=train_set, metric_adapter=nova_metric_adapter)

optimized_prompt_adapter = nova_prompt_optimizer.optimize(mode="pro")

2026/01/29 18:20:34 INFO amzn_nova_prompt_optimizer.core.optimizers.nova_prompt_optimizer.nova_prompt_optimizer: No meta_prompt_inference_adapter provided. Creating default BedrockInferenceAdapter with Nova 2.0 Lite for meta-prompting.
2026/01/29 18:20:34 INFO amzn_nova_prompt_optimizer.core.optimizers.nova_prompt_optimizer.nova_prompt_optimizer: Using separate inference adapters: Meta-prompting with BedrockInferenceAdapter, Task optimization with BedrockInferenceAdapter
2026/01/29 18:20:34 INFO amzn_nova_prompt_optimizer.core.optimizers.nova_meta_prompter.nova_mp_optimizer: Optimizing prompt using Nova Meta Prompter with Model: us.amazon.nova-2-lite-v1:0
2026/01/29 18:20:36 INFO amzn_nova_prompt_optimizer.core.optimizers.miprov2.miprov2_optimizer: Using us.amazon.nova-pro-v1:0 for Evaluation
2026/01/29 18:20:36 INFO amzn_nova_prompt_optimizer.core.optimizers.miprov2.miprov2_optimizer: Using us.amazon.nova-2-lite-v1:0 for Prompting
2026/01/29 18:20:36 INFO amzn_nova_prompt_optimizer.co

Bootstrapping set 1/20
Bootstrapping set 2/20
Bootstrapping set 3/20


  0%|          | 0/50 [00:00<?, ?it/s]/Users/shenov/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  8%|▊         | 4/50 [00:04<00:57,  1.25s/it]


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/20


  8%|▊         | 4/50 [00:04<00:46,  1.01s/it]


...

2026/01/29 18:37:12 INFO dspy.evaluate.evaluate: Average Metric: 27.99999999999999 / 35 (80.0%)
2026/01/29 18:37:12 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 80.0 on minibatch of size 35 with parameters ['Predictor 0: Instruction 5', 'Predictor 0: Few-Shot Set 5'].
2026/01/29 18:37:12 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [81.71, 79.05, 72.67, 84.1, 87.43, 75.05, 77.43, 82.38, 85.52, 84.38, 76.1, 72.0, 74.86, 87.33, 81.9, 80.57, 86.29, 85.43, 84.48, 84.86, 79.05, 76.48, 80.48, 81.81, 82.76, 85.05, 77.14, 78.95, 80.0]
2026/01/29 18:37:12 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [82.73, 82.0, 85.33, 85.4, 86.53, 86.73]
2026/01/29 18:37:12 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 86.73
2026/01/29 18:37:12 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/01/29 18:37:12 INFO dspy.teleprompt.mipro_optimizer_v2: == Trial 36 / 37 - Minibatch ==



Average Metric: 28.13 / 35 (80.4%): 100%|██████████| 35/35 [00:21<00:00,  1.65it/s]

2026/01/29 18:37:33 INFO dspy.evaluate.evaluate: Average Metric: 28.133333333333326 / 35 (80.4%)
2026/01/29 18:37:33 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 80.38 on minibatch of size 35 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 16'].
2026/01/29 18:37:33 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [81.71, 79.05, 72.67, 84.1, 87.43, 75.05, 77.43, 82.38, 85.52, 84.38, 76.1, 72.0, 74.86, 87.33, 81.9, 80.57, 86.29, 85.43, 84.48, 84.86, 79.05, 76.48, 80.48, 81.81, 82.76, 85.05, 77.14, 78.95, 80.0, 80.38]
2026/01/29 18:37:33 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [82.73, 82.0, 85.33, 85.4, 86.53, 86.73]
2026/01/29 18:37:33 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 86.73
2026/01/29 18:37:33 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2026/01/29 18:37:33 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 37 / 37 - Full Evaluation =====
2026


Average Metric: 41.13 / 50 (82.3%): 100%|██████████| 50/50 [00:29<00:00,  1.71it/s]

2026/01/29 18:38:02 INFO dspy.evaluate.evaluate: Average Metric: 41.13333333333333 / 50 (82.3%)
2026/01/29 18:38:02 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [82.73, 82.0, 85.33, 85.4, 86.53, 86.73, 82.27]
2026/01/29 18:38:02 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 86.73
2026/01/29 18:38:02 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/01/29 18:38:02 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/01/29 18:38:02 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 86.73!


In [11]:
optimized_prompt_adapter.show()

2026/01/29 18:38:02 INFO amzn_nova_prompt_optimizer.core.input_adapters.prompt_adapter: 
Standardized Prompt:


{
  "user_prompt": {
    "variables": [
      "input"
    ],
    "template": "The input provided to you is: {{input}}",
    "metadata": {
      "format": "text"
    }
  },
  "system_prompt": {
    "variables": [],
    "template": "Generate a JSON object from the provided customer support email input. The JSON must contain three keys: \"urgency\", \"sentiment\", and \"categories\".\n\n**Key Requirements:**\n1. **\"urgency\"**: Must be one of `high`, `medium`, or `low`, based on how time-sensitive the request is.\n2. **\"sentiment\"**: Must be one of `negative`, `neutral`, or `positive`, reflecting the emotional tone of the email.\n3. **\"categories\"**: Must be a dictionary where:\n   - The keys are from this fixed list: `emergency_repair_services`, `routine_maintenance_requests`, `quality_and_safety_concerns`, `specialized_cleaning_services`, `general_inquiries`, `sustainability_and_environmental_practices`, `training_and_support_requests`, `cleaning_services_scheduling`, `customer_fee

### Optimized System Prompt

In [12]:
print(optimized_prompt_adapter.system_prompt)

Generate a JSON object from the provided customer support email input. The JSON must contain three keys: "urgency", "sentiment", and "categories".

**Key Requirements:**
1. **"urgency"**: Must be one of `high`, `medium`, or `low`, based on how time-sensitive the request is.
2. **"sentiment"**: Must be one of `negative`, `neutral`, or `positive`, reflecting the emotional tone of the email.
3. **"categories"**: Must be a dictionary where:
   - The keys are from this fixed list: `emergency_repair_services`, `routine_maintenance_requests`, `quality_and_safety_concerns`, `specialized_cleaning_services`, `general_inquiries`, `sustainability_and_environmental_practices`, `training_and_support_requests`, `cleaning_services_scheduling`, `customer_feedback_and_complaints`, `facility_management_issues`.
   - Each value is a boolean (`true` or `false`) indicating whether the category best matches the content of the email. Only categories that are clearly relevant should be set to `true`; all other

### Optimized User Prompt

In [13]:
print(optimized_prompt_adapter.user_prompt)

The input provided to you is: {{input}}


### Few Shot Examples

In [14]:
print(f"Number of Few-Shot Examples = {len(optimized_prompt_adapter.few_shot_examples)}")

Number of Few-Shot Examples = 4


In [15]:
# Print only the first example
print(optimized_prompt_adapter.few_shot_examples[0])

{'input': "The input provided to you is: Subject: Inquiry About HVAC Maintenance Best Practices\n\nHi ProCare Support Team,\n\nI hope this message finds you well. My name is Alex, and I'm a novice electrician currently learning the ropes of working with HVAC units. I've recently started gaining hands-on experience and have been exploring various aspects of facility maintenance.\n\nI wanted to reach out to you with a few general inquiries regarding best practices for maintaining HVAC systems. Given your expertise in facility management and maintenance, I believe your insights would be incredibly valuable to me as I continue to develop my skills.\n\nSo far, I've been following basic maintenance routines like checking filters and ensuring proper airflow, but I would love to know if there are any specific tips or advanced techniques that I should be aware of. Additionally, are there any common pitfalls or mistakes that I should avoid while working with HVAC units?\n\nI haven't encountered 

### Evaluator

Now we evaluate the Nova Prompt Optimizer Optimized prompt

In [16]:
from amzn_nova_prompt_optimizer.core.evaluation import Evaluator

evaluator = Evaluator(optimized_prompt_adapter, test_set, metric_adapter, inference_adapter)

In [17]:
nova_prompt_optimizer_evaluation_score = evaluator.aggregate_score(model_id=NOVA_MODEL_ID)
print(f"Nova Prompt Optimizer = {nova_prompt_optimizer_evaluation_score}")

2026/01/29 18:38:02 INFO amzn_nova_prompt_optimizer.core.evaluation: Cache miss - Running new inference on Dataset
Running inference: 100%|██████████| 100/100 [00:50<00:00,  1.97it/s]
2026/01/29 18:38:53 INFO amzn_nova_prompt_optimizer.core.evaluation: Running Batch Evaluation on Dataset, using `batch_apply` metric
2026/01/29 18:38:53 INFO amzn_nova_prompt_optimizer.core.evaluation: Using cached inference results
2026/01/29 18:38:53 INFO amzn_nova_prompt_optimizer.core.evaluation: Running Evaluation on Dataset, using `apply` metric


Nova Prompt Optimizer = {'is_valid_json': 1.0, 'correct_categories': 0.912, 'correct_sentiment': 0.75, 'correct_urgency': 0.9, 'total': 0.8540000000000006}


In [18]:
optimized_prompt_adapter.save("optimized_prompt/")